# 02 CNN Model

A 1D CNN on `data/features.parquet`: predict `pfas_sum` from the distance/count features to potential PFAS sources, same target and feature set as `01_Baseline.ipynb`.

The feature vector is treated as a 1D signal (1 channel, length = n_features) so that stacked `Conv1d` layers can learn local interactions between neighbouring columns (e.g. the three distance bands of the same source type) before pooling down to a single prediction.

Every run (params, metrics, model) is tracked in MLflow. Start the MLflow server + database first (from the project root):

```bash
docker compose up -d
```

UI: http://localhost:5050

In [ ]:
import mlflow
import numpy as np
import pandas as pd
import torch
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from pfas_trace.config import DATASET_FEATURES, settings
from pfas_trace.model.cnn import Simple1DCNN

In [ ]:
mlflow.set_tracking_uri(settings.MLFLOW_TRACKING_URI)
mlflow.set_experiment("pfas-baseline")

## Load data
- **Target:** `pfas_sum` is extremely skewed (72% zeros, max ~5e8), so we train on `log1p(pfas_sum)`, same as the baseline.
- **Features:** all distance/count columns. `date`, `name`, `city` are identifiers; `lat`/`lon` are left out so the model has to learn from the PFAS sources instead of memorising locations.
- Unlike the tree-based baseline, a CNN cannot handle missing values or unscaled inputs natively, so distances/counts are median-imputed (a source being absent in range shows up as `NaN`) and standardized.

In [ ]:
df = pd.read_parquet(DATASET_FEATURES)

TARGET = "pfas_sum"
NON_FEATURES = ["date", "lat", "lon", "name", "city", TARGET]

X = df.drop(columns=NON_FEATURES)
y = np.log1p(df[TARGET])
X.shape

## Train/test split by station
A station (`name`) is measured many times but always has the same features. A random row split would put the same station in train and test and overestimate performance. `GroupShuffleSplit` keeps all measurements of a station on one side (same as the baseline).

Imputation and scaling are fit on the train split only, to avoid leaking test-set statistics into the model.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df["name"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

preprocessor = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

X_train_scaled = preprocessor.fit_transform(X_train).astype(np.float32)
X_test_scaled = preprocessor.transform(X_test).astype(np.float32)

len(X_train), len(X_test)

## Train and log
One helper that fits the `Simple1DCNN` with mini-batch gradient descent, evaluates on the held-out stations, and logs params, metrics and the model itself to MLflow. Metrics are on the log scale, same as the baseline, so the two are directly comparable.

Training uses a validation split (also by station) for early stopping, to avoid picking an overfit epoch.

In [ ]:
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Hold out a validation set from the training stations for early stopping.
val_splitter = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
fit_idx, val_idx = next(
    val_splitter.split(X_train_scaled, y_train, groups=df["name"].iloc[train_idx])
)


def to_tensor_dataset(features: np.ndarray, target: pd.Series) -> TensorDataset:
    return TensorDataset(
        torch.from_numpy(features),
        torch.tensor(target.to_numpy(dtype=np.float32)),
    )


fit_loader = DataLoader(
    to_tensor_dataset(X_train_scaled[fit_idx], y_train.iloc[fit_idx]),
    batch_size=512,
    shuffle=True,
)
val_loader = DataLoader(
    to_tensor_dataset(X_train_scaled[val_idx], y_train.iloc[val_idx]),
    batch_size=2048,
)
test_loader = DataLoader(
    to_tensor_dataset(X_test_scaled, y_test),
    batch_size=2048,
)


@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    predictions = [model(features.to(DEVICE)).cpu() for features, _ in loader]
    return torch.cat(predictions).numpy()


def train_and_log(
    params: dict,
    run_name: str,
    max_epochs: int = 50,
    patience: int = 5,
) -> dict:
    model = Simple1DCNN(n_features=X_train_scaled.shape[1], **params).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    best_val_rmse = float("inf")
    best_state = model.state_dict()
    epochs_without_improvement = 0

    with mlflow.start_run(run_name=run_name):
        for epoch in range(max_epochs):
            model.train()
            for features, targets in fit_loader:
                features, targets = features.to(DEVICE), targets.to(DEVICE)

                optimizer.zero_grad()
                loss = loss_fn(model(features), targets)
                loss.backward()
                optimizer.step()

            val_rmse = root_mean_squared_error(
                y_train.iloc[val_idx], predict(model, val_loader)
            )
            mlflow.log_metric("val_rmse", val_rmse, step=epoch)

            if val_rmse < best_val_rmse:
                best_val_rmse = val_rmse
                best_state = model.state_dict()
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if epochs_without_improvement >= patience:
                break

        model.load_state_dict(best_state)
        y_pred = predict(model, test_loader)

        metrics = {
            "rmse": root_mean_squared_error(y_test, y_pred),
            "mae": mean_absolute_error(y_test, y_pred),
            "r2": r2_score(y_test, y_pred),
            "best_val_rmse": best_val_rmse,
            "epochs_trained": epoch + 1,
        }
        mlflow.log_params({**params, "max_epochs": max_epochs, "patience": patience})
        mlflow.log_metrics(metrics)
        mlflow.pytorch.log_model(model, name="model")

    return metricsll

**CNN:** default architecture from `Simple1DCNN` (three Conv1d blocks, 32/64/64 channels). Compare `rmse`/`mae`/`r2` against the `dummy-mean` and `hist-gradient-boosting` runs from `01_Baseline.ipynb` in the MLflow UI.

In [ ]:
train_and_log({}, "simple-1d-cnn")